# 📝 과제 02: 의약품 상담 도우미에 LightRAG 적용하기

## 만들 결과

동네 약국의 **상담 도우미 검색기**를 만듭니다. 손님 질문에 답할 근거 안내문을 찾고, 청크만 찾는 `naive`와 개체·관계까지 쓰는 `mix` 가운데 어느 쪽이 근거를 더 찾는지 비교합니다.

- 제품·성분·제조사·증상을 개체로 뽑도록 LightRAG를 설정하고 안내문 18편을 색인합니다.
- 골드 질문 8개를 두 모드로 검색·답변하고 결과 표를 `output/gwaje02_results.csv`에 저장합니다.
- 새 안내문 1편을 추가하고, 취급을 멈춘 제품의 안내문 1편을 지운 뒤 그 제품을 묻던 질문을 다시 검색합니다.

교안 02의 색인, 검색 모드, 문서 추가·삭제를 적용합니다. 의학적 조언이 아닌 수업용 검색 실습이며, 실제 복용은 의사·약사와 상의해야 합니다.

<img src="images/gwaje02_overview.png" width="1050" alt="의약품 안내문 18편을 LightRAG로 색인해 제품·성분·제조사·증상 개체와 관계 그래프를 만들고, 골드 질문을 naive(청크 검색)와 mix(개체·관계와 청크 검색)로 검색해 근거 문서를 비교합니다. 이어서 더모픽스크림 안내문을 추가하고 알레크롬점안액 안내문을 지운 뒤 M1을 다시 검색합니다.">

## 준비: 데이터와 제공 코드

- **자료:** `data/과제_02/`의 의약품 안내문 19편(색인 18편, 추가용 1편), 골드 질문 8개(단일 사실 3, 멀티홉 3, 전역 2), 추가·삭제할 문서를 정한 `extra.json`. 안내문 출처는 식품의약품안전처 「의약품개요정보(e약은요)」(공공누리 출처표시)입니다.
- **순서:** 개체 유형 안내 -> LightRAG 설정 -> 근거 문서 ID 모으기 -> 색인 -> 모드 비교 -> 문서 추가 -> 문서 삭제 -> 서술
- **제공:** 환경 설정, 자료 읽기, 모델 함수(`course_llm`, `embedding_func`)와 호출 계수기 `tracker`, `read_graph`, `evidence_hit`, `make_result_row`, 결과를 읽는 셀
- **작성:** `[작성]` 셀의 `...`를 번호 주석대로 채우고 바로 아래 자가채점 셀을 실행하세요. 자가채점은 모델을 부르지 않습니다. 모델은 색인부터 실제로 부릅니다.

In [ ]:
# 실습에 필요한 라이브러리를 불러옵니다. 나머지는 처음 쓰는 셀에서 불러옵니다.
import json
from pathlib import Path

import pandas as pd
from dotenv import find_dotenv, load_dotenv

material_dir = Path(".").resolve()
data_dir = material_dir / "data"
output_dir = material_dir / "output"
output_dir.mkdir(parents=True, exist_ok=True)

# 노트북 폴더부터 위로 올라가며 처음 찾은 .env를 읽습니다.
load_dotenv(find_dotenv(usecwd=True))
# 단원 .env에 없는 키는 저장소 최상위 .env에서 읽습니다(이미 읽은 키는 유지).
load_dotenv(material_dir.parent / ".env")

In [ ]:
def read_json(name):
    """data 폴더 아래 name 경로의 JSON 파일을 읽어 list 또는 dict로 반환합니다.

    name: data 폴더 기준 상대 경로(예: "과제_02/questions.json")
    """
    return json.loads((data_dir / name).read_text(encoding="utf-8"))


gwaje_dir = data_dir / "과제_02"
# 추가·삭제할 문서: insert는 색인 뒤에 넣을 문서, delete는 색인한 뒤 지울 문서와 답이 바뀌는 질문입니다.
extra = read_json("과제_02/extra.json")
# 추가용 1편은 색인할 문서 목록에서 빼야 하므로, 그 문서 ID를 여기서 먼저 읽습니다.
insert_id = extra["insert"]["doc_id"]

# 색인할 안내문 18편: 파일 이름이 "<문서 ID>.txt"이고 첫 줄이 제품명입니다.
doc_paths = []

for path in sorted((gwaje_dir / "docs").glob("*.txt")):
    # path.stem은 확장자를 뺀 파일 이름(문서 ID)입니다.
    if path.stem != insert_id:
        doc_paths.append(path)

# 골드 질문마다 id, qtype, question, gold_doc_ids(정답 근거 문서 ID)가 있습니다.
questions = read_json("과제_02/questions.json")
questions_by_id = {question["id"]: question for question in questions}

TOP_K = 5  # 답변 모델에 넘길 청크 수
# 답 형식 지시: 전역 질문은 항목을 모두 나열하게, 나머지는 짧게 답하게 합니다.
SHORT_ANSWER = "한국어로 한두 문장. 근거에 없으면 모른다고 답한다"
LIST_ANSWER = "한국어로, 해당하는 항목을 모두 나열한 목록"


def evidence_hit(given_ids, gold_ids):
    """정답 근거 문서 가운데 검색이 넘긴 문서가 몇 편인지 세어 int로 반환합니다.

    given_ids: 검색이 답변 모델에 넘긴 원문 문서 ID 목록
    gold_ids: 질문의 정답 근거 문서 ID 목록(골드 질문의 gold_doc_ids)
    예: 정답 근거 3편 가운데 2편을 넘겼으면 2를 반환합니다.
    """
    hit = 0

    # 정답 근거 문서마다 넘긴 목록에 들어 있는지 봅니다.
    for gold_id in gold_ids:
        if gold_id in given_ids:
            hit += 1

    return hit


def make_result_row(method, question, given_ids, usage, seconds):
    """검색 한 번의 결과를 비교용 결과 표의 한 행(dict)으로 만들어 반환합니다.

    method: 검색 방식 이름(예: "graphrag_local", "lightrag_mix")
    question: 골드 질문 dict(id, qtype, gold_doc_ids)
    given_ids: 검색이 답변 모델에 넘긴 원문 문서 ID 목록. 원문을 넘기지 않는 GraphRAG Global은 None을 넘기고, 이때 문서 열은 빈칸(None)이 됩니다.
    usage: 이 검색의 LLM 사용량 dict(llm_calls 호출 수, input_tokens 입력 토큰, output_tokens 출력 토큰)
    seconds: 검색에 걸린 시간(초)
    문서 열: docs_given(넘긴 문서 수), gold_docs(정답 근거 수), hit_docs(정답 근거 가운데 넘긴 수), all_hit(모두 넘겼으면 1, 아니면 0)
    """
    # 사용량과 시간은 모든 방식에 채우고, 문서 열은 빈칸(None)으로 시작합니다.
    row = {"method": method, "question_id": question["id"], "qtype": question["qtype"],
           "docs_given": None, "gold_docs": None, "hit_docs": None, "all_hit": None,
           "llm_calls": usage["llm_calls"], "input_tokens": usage["input_tokens"],
           "output_tokens": usage["output_tokens"], "seconds": round(seconds, 1)}

    # 원문 문서를 넘긴 방식만 문서 열을 채웁니다.
    if given_ids is not None:
        gold_ids = question["gold_doc_ids"]
        hit = evidence_hit(given_ids, gold_ids)
        row["docs_given"] = len(given_ids)
        row["gold_docs"] = len(gold_ids)
        row["hit_docs"] = hit

        # 정답 근거를 모두 넘겼으면 1
        if hit == len(gold_ids):
            row["all_hit"] = 1
        else:
            row["all_hit"] = 0

    return row


print("색인할 안내문:", len(doc_paths), "편 | 추가할 문서 ID:", insert_id, "| 골드 질문:", len(questions), "개")

In [ ]:
# 색인할 안내문의 제품명(파일 첫 줄)을 확인합니다.
for path in doc_paths:
    print(path.stem, "|", path.read_text(encoding="utf-8").splitlines()[0])

In [ ]:
# 손님 질문 8개를 확인합니다. 멀티홉 질문에는 다리(제조사, 함께 쓰는 약의 성분)와 답이 든 제품 이름이 없습니다.
for question in questions:
    print(question["id"], question["qtype"], "| 근거 문서", len(question["gold_doc_ids"]), "편")
    print("   ", question["question"])

In [ ]:
# 답변 모델·임베딩 모델과 LightRAG에 넘길 모델 함수
import os

import numpy as np
from langchain_openai import OpenAIEmbeddings
from lightrag.llm.openai import openai_complete_if_cache
from lightrag.utils import EmbeddingFunc, TokenTracker

# 답변 모델(.env에 없으면 gpt-5.6-luna)과 임베딩 모델(교안 02와 같은 768차원)
model_name = os.getenv("OPENAI_CHAT_MODEL", "gpt-5.6-luna")
embedding_model = OpenAIEmbeddings(model="text-embedding-3-large", dimensions=768)

# LightRAG 색인 파일을 저장할 폴더와 이 과제의 workspace(하위 폴더) 이름
lightrag_dir = output_dir / "lightrag"
WORKSPACE = "gwaje02"
# LLM 호출 수와 토큰 수를 더해 두는 계수기. 잴 구간 앞에서 reset()으로 0으로 되돌립니다.
tracker = TokenTracker()


async def course_llm(prompt, **kwargs):
    """prompt를 답변 모델(model_name)에 보내 응답 글(str)을 반환하고, 호출 수와 토큰 수를 tracker에 더합니다.

    LightRAG가 개체·관계 추출, 설명 요약, 질문 키워드 추출, 답변 작성에 부르는 LLM 함수입니다.
    prompt: LightRAG가 만든 프롬프트 글
    kwargs: LightRAG가 함께 넘기는 system_prompt 등. 그대로 openai_complete_if_cache에 전달합니다.
    """
    # openai_complete_if_cache는 LightRAG가 제공하는 OpenAI 호출 함수입니다.
    # LightRAG가 넘기는 나머지 인자(system_prompt 등)는 그대로 전달합니다.
    return await openai_complete_if_cache(model_name, prompt, token_tracker=tracker, **kwargs)


async def course_embed(texts):
    """texts의 글을 한 번에 임베딩해 (글 수, 768) 모양의 numpy 배열로 반환합니다.

    LightRAG가 청크·개체·관계를 벡터 저장소에 넣을 때와 질문 키워드로 검색할 때 부릅니다.
    texts: 임베딩할 글(str) 목록
    """
    # LangChain 임베딩은 벡터 리스트를 돌려주므로 LightRAG가 받는 numpy 배열로 바꿉니다.
    return np.array(await embedding_model.aembed_documents(texts))


# LightRAG는 벡터 저장소를 만들 때 차원이 필요해, 함수를 차원(768)과 함께 EmbeddingFunc로 감쌉니다.
embedding_func = EmbeddingFunc(embedding_dim=768, func=course_embed)

print("답변 모델:", model_name)

## 1. 의약품에 맞는 개체 유형 안내를 정하세요

LightRAG는 `addon_params["entity_types_guidance"]` 문장을 개체·관계 추출 프롬프트에 넣어, 뽑은 개체를 어떤 유형으로 나눌지 정합니다. 교안의 관광지 색인은 `"장소, 인물, 단체, 행사 중 하나로 분류한다. 맞는 것이 없으면 Other."`를 썼습니다. 의약품의 멀티홉 질문은 제조사나 성분을 거쳐 다른 제품으로 넘어가므로, 이 둘이 개체로 뽑혀야 합니다.

**작성:** 개체 유형 안내 문장 `DRUG_ENTITY_GUIDANCE`

**목적:** 교안의 개체 유형 안내를 의약품에 맞게 바꿔, 같은 제조사·같은 성분의 제품이 그래프에서 한 개체로 이어지게 합니다.

**구현할 처리**
1. 교안 문장처럼 한국어 한두 문장(200자 이내)으로 씁니다.
2. 유형 이름 `제품`, `성분`, `제조사`, `증상`을 모두 넣습니다. 필요하면 한두 개를 더 넣어도 됩니다.
3. 맞는 유형이 없으면 `Other`로 분류하라는 말을 붙입니다. `Other`는 영어 그대로 씁니다.

**반환값:** 이 셀이 만드는 변수 `DRUG_ENTITY_GUIDANCE`(str)

In [ ]:
# [작성] 의약품 안내문에 맞는 개체 유형 안내를 정하세요.
# 1~3. 제품·성분·제조사·증상 유형과 Other 규칙을 담은 한국어 문장을 DRUG_ENTITY_GUIDANCE에 담으세요.
...

In [ ]:
# [자가채점] 개체 유형 안내가 문자열이고 필요한 유형과 Other 규칙을 담았는지 확인합니다.
assert "DRUG_ENTITY_GUIDANCE" in globals() and isinstance(DRUG_ENTITY_GUIDANCE, str), "DRUG_ENTITY_GUIDANCE에 문자열을 담으세요. `...`를 채웠는지도 확인하세요."
assert any("가" <= char <= "힣" for char in DRUG_ENTITY_GUIDANCE), "안내문은 한국어로 쓰세요."

for entity_type in ["제품", "성분", "제조사", "증상"]:
    assert entity_type in DRUG_ENTITY_GUIDANCE, f"유형 '{entity_type}'을(를) 안내문에 넣으세요."

assert "Other" in DRUG_ENTITY_GUIDANCE, "맞는 유형이 없을 때 Other로 분류하라는 말을 붙이세요."
assert len(DRUG_ENTITY_GUIDANCE) <= 200, "안내문은 한두 문장으로 짧게 쓰세요. 추출 프롬프트에 그대로 들어갑니다."

print("개체 유형 안내문 확인 완료:", DRUG_ENTITY_GUIDANCE)

## 2. LightRAG를 의약품 색인용으로 설정하세요

교안의 관광지 색인 셀과 같은 인자로 `LightRAG`를 만들고, `workspace`(이 과제의 저장 폴더)와 개체 유형 안내만 바꿉니다. 이 셀은 모델을 부르지 않습니다. 문서 색인은 뒤의 색인 셀에서 따로 합니다.

**작성:** 의약품 색인용 `LightRAG` 객체 `rag`

**목적:** 교안에서 배운 모델 함수 연결과 `addon_params` 한국어 설정을 의약품 자료에 적용합니다.

**구현할 처리**
1. 아래 표의 인자로 `LightRAG(...)`를 만들어 `rag`에 담습니다.
2. `await rag.initialize_storages()`로 그래프·벡터·문서 상태 저장소를 엽니다. 색인·검색 전에 한 번 부릅니다.

| 인자 | 넣을 값 |
|---|---|
| `working_dir` | `str(lightrag_dir)` |
| `workspace` | `WORKSPACE` |
| `llm_model_func` | `course_llm` |
| `embedding_func` | `embedding_func` |
| `addon_params` | `{"language": "Korean", "entity_types_guidance": DRUG_ENTITY_GUIDANCE}` |
| `entity_extract_max_gleaning` | `0` (추출을 한 번만 합니다) |
| `enable_llm_cache` | `False` (검색의 키워드 추출·답변 응답 캐시를 꺼 질문마다 호출을 셉니다. 추출 응답 캐시는 남아 문서 삭제에 쓰입니다) |

**반환값:** 이 셀이 만드는 변수 `rag`(저장소를 연 `LightRAG` 객체)

In [ ]:
# [작성] 의약품 색인용 LightRAG를 만들고 저장소를 여세요.
import shutil
from lightrag import LightRAG

# 이전 색인 폴더를 통째로 지웁니다. 남아 있으면 이미 넣은 문서로 보고 추출하지 않습니다.
# ignore_errors=True면 폴더가 없어도 오류가 나지 않습니다.
shutil.rmtree(lightrag_dir / WORKSPACE, ignore_errors=True)

# 1. 설명에 나온 설정대로 LightRAG를 만들어 rag에 담으세요.
...

# 2. 그래프·벡터·문서 상태 저장소를 여세요.
...

In [ ]:
# [자가채점] 만든 객체의 설정값을 읽어 확인합니다. 모델은 호출하지 않습니다.
assert "rag" in globals() and isinstance(rag, LightRAG), "LightRAG(...)로 만든 객체를 rag에 담으세요. `...`를 채웠는지도 확인하세요."
assert Path(rag.working_dir) == lightrag_dir and rag.workspace == WORKSPACE, "working_dir에는 str(lightrag_dir), workspace에는 WORKSPACE를 넣으세요."
assert rag.llm_model_func is course_llm, "llm_model_func에는 계수기를 쓰는 course_llm을 넣으세요."
assert rag.embedding_func.embedding_dim == 768, "embedding_func에는 768차원 embedding_func를 넣으세요."
assert rag.addon_params.get("language") == "Korean", "addon_params의 language를 'Korean'으로 정하세요. 기본값은 English입니다."
assert rag.addon_params.get("entity_types_guidance") == DRUG_ENTITY_GUIDANCE, "addon_params의 entity_types_guidance에 DRUG_ENTITY_GUIDANCE를 넣으세요."
assert rag.entity_extract_max_gleaning == 0, "entity_extract_max_gleaning=0으로 추출을 한 번만 하게 하세요."
assert rag.enable_llm_cache is False, "enable_llm_cache=False로 검색 응답 캐시를 끄세요. 켜 두면 같은 질문을 다시 검색할 때 호출 수를 제대로 셀 수 없습니다."
assert rag._storages_status == "initialized", "await rag.initialize_storages()로 저장소를 여세요."

print("LightRAG 설정 확인 완료")

## 3. 검색 결과에서 근거 문서 ID를 모으세요

교안의 검색 모드 비교에서 본 것처럼, `aquery_data`와 `aquery_llm`의 결과 dict에는 답변 모델에 넘긴 개체·관계·청크가 `data["data"]`에 들어 있습니다. 색인할 때 파일 이름 `<문서 ID>.txt`를 `file_paths`로 넘기므로, 청크의 `file_path`에서 `.txt`를 떼면 문서 ID가 됩니다. 개체·관계에도 `file_path`가 있지만 세지 않습니다. 그 설명은 여러 청크를 합친 LLM 요약이라 원문 근거가 아니기 때문입니다.

```python
{"data": {"entities": [{"entity_name": ..., "file_path": ...}, ...],
          "relationships": [...],
          "chunks": [{"chunk_id": ..., "file_path": "drug_....txt", "content": ...}, ...]},
 "metadata": {...}}
```

**작성:** `chunk_doc_ids(data)` 함수

**목적:** 교안의 근거 확인 방법대로 답변 모델에 넘긴 원문 청크를 문서 ID로 바꿔, 정답 근거 문서를 넘겼는지 셀 수 있게 합니다.

**구현할 처리**
1. `data["data"]["chunks"]`의 청크마다 `file_path`에서 `.txt`를 떼어 문서 ID를 얻습니다.
2. 같은 문서의 청크가 여러 개여도 문서 ID는 한 번만 남깁니다.
3. 문서 ID를 정렬한 리스트로 반환합니다. 입력 `data`는 바꾸지 않습니다.

**반환값:** 문서 ID 리스트(`list[str]`). 청크가 없으면 빈 리스트입니다. 예: 청크 셋의 `file_path`가 `drug_199801231.txt`, `drug_195900034.txt`, `drug_199801231.txt`이면 `["drug_195900034", "drug_199801231"]`

In [ ]:
# [작성] 검색 결과의 청크가 나온 문서 ID를 중복 없이 정렬해 반환하세요.
def chunk_doc_ids(data):
    """aquery_data 검색 결과에서 답변 모델에 넘길 원문 청크가 나온 문서 ID를 중복 없이 정렬한 list로 반환합니다.

    data: rag.aquery_data(...)가 반환한 dict. 개체·관계는 LLM이 쓴 설명이라 세지 않고 원문 청크만 봅니다. 청크가 없으면 빈 list를 반환합니다.
    """
    # 1~2. 청크마다 file_path에서 .txt를 뗀 문서 ID를 중복 없이 모으세요.
    ...

    # 3. 정렬한 목록을 반환하세요.
    ...

In [ ]:
# [자가채점] 고정된 검색 결과 dict로 확인합니다. 모델은 호출하지 않습니다.
from copy import deepcopy

sample = {
    "status": "success",
    "data": {
        # 개체 설명은 LLM 요약이라 근거로 세지 않습니다. 이 개체에만 나온 문서는 결과에 들어가면 안 됩니다.
        "entities": [{"entity_name": "이연제약", "file_path": "drug_199700265.txt<SEP>drug_197900219.txt"}],
        "relationships": [],
        "chunks": [
            {"chunk_id": "chunk-a", "file_path": "drug_199700265.txt", "content": "알레리진정"},
            {"chunk_id": "chunk-b", "file_path": "drug_198100234.txt", "content": "제이알히드로코르티손연고"},
            {"chunk_id": "chunk-c", "file_path": "drug_199700265.txt", "content": "알레리진정 용법"},
        ],
        "references": [],
    },
    "metadata": {"query_mode": "mix"},
}
before = deepcopy(sample)
result = chunk_doc_ids(sample)
assert isinstance(result, list), "목록(list)을 반환하세요. `...`를 채웠는지도 확인하세요."
assert all(not doc_id.endswith(".txt") for doc_id in result), "file_path에서 .txt를 떼어 문서 ID만 남기세요."
assert len(result) == len(set(result)), "같은 문서의 청크가 여러 개여도 문서 ID는 한 번만 남기세요."
assert result == sorted(result), "문서 ID를 정렬해 반환하세요."
assert "drug_197900219" not in result, "개체의 file_path가 아니라 청크(data['data']['chunks'])의 file_path만 쓰세요."
assert set(result) == {"drug_199700265", "drug_198100234"}, "청크가 나온 문서를 빠짐없이 모으세요."
assert sample == before, "입력 dict는 바꾸지 마세요."
empty = {"data": {"entities": [], "relationships": [], "chunks": []}}
assert chunk_doc_ids(empty) == [], "청크가 없으면 빈 목록을 반환하세요."

print("근거 문서 ID 확인 완료")

## 4. 안내문 18편을 색인하세요

여기부터 모델을 실제로 부르며, 색인에 몇 분 걸립니다. 교안의 관광지 색인은 `file_paths`만 넘겼지만, 이 과제는 뒤에서 안내문 한 편을 문서 ID로 지우므로 `ids`도 함께 넘깁니다. `ids`를 넘기지 않으면 LightRAG가 파일 이름을 해시해 `doc-`로 시작하는 ID를 만들어, `extra.json`에 적힌 문서 ID로는 지울 수 없습니다.

**작성:** 안내문 18편을 문서 ID·파일 이름과 함께 색인하는 코드

**목적:** 교안의 색인처럼 개체·관계 그래프를 만들고, 색인에 든 LLM 호출 수를 잽니다.

**구현할 처리**
1. `doc_paths`(안내문 파일 경로 18개)로 같은 순서의 목록 세 개를 만듭니다. 본문 `texts`, 문서 ID `doc_ids`(파일 이름에서 `.txt`를 뗀 것, 예: `drug_199801231`), 파일 이름 `file_paths`(예: `drug_199801231.txt`)입니다.
2. `tracker.reset()`으로 계수기를 0으로 되돌리고, 시간을 재며 `await rag.ainsert(texts, ids=doc_ids, file_paths=file_paths)`로 색인합니다.
3. 색인 호출 수 `tracker.call_count`를 `index_calls`에, 걸린 초를 `index_seconds`에 담습니다.

**반환값:** 이 셀이 만드는 변수 `index_calls`(int), `index_seconds`(float)

문서 ID나 파일 이름이 이미 있는 문서는 건너뜁니다. 같은 커널에서는 색인 폴더를 지워도 메모리에 기록이 남으므로, 색인을 다시 하려면 커널을 다시 시작하고 처음부터 실행하세요.

In [ ]:
# 색인 결과 그래프 파일을 읽는 함수
import networkx as nx


def read_graph(workspace):
    """LightRAG가 저장한 지식 그래프 파일(graphml)을 읽어 NetworkX 그래프로 반환합니다.

    workspace: 색인할 때 LightRAG에 준 workspace 이름(예: WORKSPACE의 값 "gwaje02"). lightrag_dir/<workspace>/graph_chunk_entity_relation.graphml 파일을 읽습니다.
    노드 이름이 개체 이름이고, 관계(edge) 속성에 keywords와 description이 있습니다.
    """
    # LightRAG는 개체·관계 그래프를 graphml 파일로 저장하고, 색인·추가·삭제 때마다 갱신합니다.
    return nx.read_graphml(lightrag_dir / workspace / "graph_chunk_entity_relation.graphml")

In [ ]:
# [작성] 안내문 18편을 문서 ID·파일 이름과 함께 색인하세요.
import time

# 1. 본문 목록 texts, 문서 ID 목록 doc_ids, 파일 이름 목록 file_paths를 만드세요.
...

# 2. 계수기를 0으로 되돌리고 시간을 재며 ids와 file_paths를 함께 넘겨 색인하세요.
...

# 3. 색인 호출 수를 index_calls에, 걸린 초를 index_seconds에 담으세요.
...

In [ ]:
# [자가채점] 문서 상태 저장소와 그래프 파일로 색인 결과를 확인합니다. 모델은 호출하지 않습니다.
expected_ids = [path.stem for path in doc_paths]
statuses = await rag.aget_docs_by_ids(expected_ids)
assert len(statuses) == len(expected_ids), "문서 ID로 찾을 수 없는 안내문이 있습니다. ainsert에 ids=doc_ids(파일 이름에서 .txt를 뗀 것)를 넘기고, 커널을 다시 시작해 처음부터 실행하세요."

for doc_id in expected_ids:
    assert statuses[doc_id]["status"] == "processed", f"{doc_id} 색인이 끝나지 않았습니다. API 키와 셀 출력의 오류를 확인하고, 커널을 다시 시작해 처음부터 실행하세요."
    assert statuses[doc_id]["file_path"] == f"{doc_id}.txt", "file_paths에 '<문서 ID>.txt' 파일 이름을 넘기세요. 근거 문서를 셀 때 씁니다."

assert await rag.aget_docs_by_ids(insert_id) == {}, "추가용 문서는 아직 넣지 않습니다. doc_paths만 색인하세요."
assert "index_calls" in globals() and index_calls >= len(expected_ids), "tracker.reset() 뒤에 색인하고 tracker.call_count를 index_calls에 담으세요. 문서마다 추출을 한 번씩 부릅니다."
assert "index_seconds" in globals() and index_seconds > 0, "색인에 걸린 초를 index_seconds에 담으세요."
graph = read_graph(WORKSPACE)
assert graph.number_of_nodes() > 0, "그래프에 개체가 없습니다. 색인 셀의 오류를 확인하세요."

print("색인 확인 완료 | 호출:", index_calls, "회 | 입력 토큰:", tracker.prompt_tokens, "| 출력 토큰:", tracker.completion_tokens, "| 걸린 시간:", round(index_seconds), "초")
print("개체:", graph.number_of_nodes(), "개 | 관계:", graph.number_of_edges(), "개")

개체 유형 안내대로 뽑혔는지 유형별 개체 수를 봅니다.

In [ ]:
# 개체 유형별 개체 수
graph = read_graph(WORKSPACE)
# LightRAG는 개체(노드)마다 추출한 유형을 entity_type 속성에 저장합니다.
entity_types = [node["entity_type"] for _, node in graph.nodes(data=True)]

display(pd.Series(entity_types).value_counts().rename("개체 수").to_frame())

M1의 다리인 이연제약 개체에 이어진 관계입니다. 알레리진정 말고 다른 제품도 이 개체에 이어졌는지 보세요. 관계는 방향 없이 저장되므로 `source`·`target`은 저장 순서일 뿐입니다.

In [ ]:
# 그래프에서 이름에 이연제약이 들어간 개체에 이어진 관계만 모아 봅니다.
relation_rows = []

# 관계(엣지)에는 종류 이름 대신 색인 때 LLM이 붙인 키워드(keywords)와 설명 문장(description)이 있습니다.
for source, target, edge in graph.edges(data=True):
    if "이연제약" in source or "이연제약" in target:
        relation_rows.append({"source": source, "target": target,
                              "keywords": edge["keywords"], "description": edge["description"]})

display(pd.DataFrame(relation_rows))

## 5. 골드 질문 8개를 naive와 mix로 비교하세요

교안에서 본 대로 `naive`는 질문을 그대로 임베딩해 원문 청크만 찾고, 키워드를 뽑지 않아 LLM은 답변 1회만 부릅니다. `mix`는 먼저 LLM이 질문에서 구체 키워드와 주제 키워드를 뽑고(1회), 구체 키워드로 개체를, 주제 키워드로 관계를 찾은 뒤 질문 임베딩으로 찾은 청크를 더해 넘깁니다. `aquery_data`는 여기까지 검색만 하고, `aquery_llm`은 같은 검색 뒤 같은 `course_llm`으로 답변까지 만듭니다. 이 과제는 답변도 읽으므로 `aquery_llm`을 씁니다.

**작성:** 골드 질문마다 두 모드로 검색·답변하고 결과 표를 저장하는 코드

**목적:** 교안의 M1 답변 생성 셀을 질문 8개 x 모드 2개로 넓혀, 그래프를 쓰는 `mix`가 질문 유형별로 정답 근거를 더 넘기는지와 그 비용을 비교합니다.

**구현할 처리**
1. 질문마다 답 형식을 고릅니다. 전역 질문(`question["qtype"] == "global"`)은 `LIST_ANSWER`, 나머지는 `SHORT_ANSWER`입니다.
2. 모드마다 `QueryParam(mode=mode, chunk_top_k=TOP_K, enable_rerank=False, response_type=답 형식)`을 만듭니다. 교안의 M1 검색과 같은 조건(청크 `TOP_K`개, 리랭킹 끔)입니다.
3. `tracker.reset()` 뒤 시간을 재며 `result = await rag.aquery_llm(question["question"], param=param)`을 부릅니다.
4. 사용량 `usage = {"llm_calls": tracker.call_count, "input_tokens": tracker.prompt_tokens, "output_tokens": tracker.completion_tokens}`를 만들고, `make_result_row("lightrag_" + mode, question, chunk_doc_ids(result), usage, 걸린 초)`를 `rows`에 더합니다.
5. 답변 문장 `result["llm_response"]["content"]`를 `answers[(question["id"], mode)]`에 담습니다.
6. 반복이 끝나면 `rows`를 DataFrame으로 만들어 `output_dir / "gwaje02_results.csv"`에 `index=False`로 저장합니다.

**반환값:** 이 셀이 만드는 변수 `rows`(결과 행 16개: 질문 8개 x 모드 2개), `answers`(답변 16개)와 CSV 파일

In [ ]:
# [작성] 골드 질문마다 naive와 mix로 검색·답변하고 결과 표를 저장하세요.
from lightrag import QueryParam

MODES = ["naive", "mix"]
rows = []
answers = {}

for question in questions:
    # 1. 전역 질문이면 LIST_ANSWER, 나머지는 SHORT_ANSWER를 답 형식으로 고르세요.
    ...

    for mode in MODES:
        # 2. 모드, TOP_K, 리랭킹 끔, 답 형식으로 QueryParam을 만드세요.
        ...

        # 3. 계수기를 0으로 되돌리고 시간을 재며 aquery_llm으로 검색·답변하세요.
        ...

        # 4. 사용량 dict와 chunk_doc_ids로 결과 한 행을 만들어 rows에 더하세요. method는 "lightrag_" + mode입니다.
        ...

        # 5. 답변 문장을 answers[(질문 ID, mode)]에 담으세요.
        ...

# 6. rows를 output/gwaje02_results.csv로 저장하세요.
...

In [ ]:
# [자가채점] 결과 표와 저장한 CSV를 확인합니다. 모델은 호출하지 않습니다.
assert "rows" in globals() and len(rows) == len(questions) * 2, "질문 8개 x 모드 2개, 16행을 rows에 담으세요."
pairs = set()

for row in rows:
    pairs.add((row["question_id"], row["method"]))
    question = questions_by_id[row["question_id"]]
    assert row["qtype"] == question["qtype"] and row["gold_docs"] == len(question["gold_doc_ids"]), "make_result_row에 그 질문의 dict를 넘기세요."
    assert row["docs_given"] <= TOP_K, f"{row['question_id']} {row['method']}: 넘긴 문서가 TOP_K보다 많습니다. QueryParam에 chunk_top_k=TOP_K를 넣으세요."
    assert 1 <= row["llm_calls"] <= 3, f"{row['question_id']} {row['method']}: 질문마다 tracker.reset()을 하고 그 질문의 호출만 세세요."

expected_pairs = {(question["id"], "lightrag_" + mode) for question in questions for mode in ["naive", "mix"]}
assert pairs == expected_pairs, "method는 'lightrag_naive', 'lightrag_mix'이고, 질문마다 두 모드를 한 번씩 실행하세요."
assert "answers" in globals() and len(answers) == len(rows), "answers[(질문 ID, mode)]에 답변 16개를 담으세요."

for key in answers:
    assert isinstance(answers[key], str) and answers[key].strip(), f"{key}: result['llm_response']['content']의 답변 문장을 담으세요."

saved = pd.read_csv(output_dir / "gwaje02_results.csv")
assert len(saved) == len(rows) and list(saved.columns) == list(rows[0].keys()), "rows를 index=False로 gwaje02_results.csv에 저장하세요."

print("모드 비교 결과 확인 완료")

질문 유형별로 두 모드의 근거 문서 포함률(`hit_rate` = `hit_docs`/`gold_docs`), 근거를 모두 넘긴 질문 비율(`all_hit`), 질문 1개당 LLM 호출과 입력 토큰을 봅니다.

In [ ]:
# 질문 유형 x 모드별 평균
results = pd.DataFrame(rows)
# 근거 문서 포함률: 정답 근거 문서 가운데 답변 모델에 넘긴 문서의 비율
results["hit_rate"] = results["hit_docs"] / results["gold_docs"]
summary = results.groupby(["qtype", "method"])[["hit_rate", "all_hit", "llm_calls", "input_tokens"]].mean().round(2)

display(summary)

멀티홉 질문 M1의 `mix` 답변입니다. 알레리진정의 제조사를 거쳐 같은 회사의 다른 제품과 용법까지 답하는지 보세요. 문서를 지운 뒤 이 답과 비교합니다.

In [ ]:
# M1 mix 답변
from IPython.display import Markdown

display(Markdown(answers[("M1", "mix")]))

## 6. 새 안내문을 추가하세요

약국이 무좀 크림 더모픽스크림을 새로 들였습니다. 교안의 문서 추가처럼 같은 `rag`에 한 편만 넣으면, LightRAG는 새 문서만 추출하고 이름이 같은 개체는 기존 노드에 합칩니다.

**작성:** 새 안내문을 문서 ID와 함께 추가하고 새로 생긴 개체를 찾는 코드

**목적:** 교안의 문서 추가를 적용해, 처음 색인보다 호출이 얼마나 적은지와 새 제품이 기존 그래프의 어떤 개체에 붙는지 확인합니다.

**구현할 처리**
1. 추가하기 전에 `read_graph(WORKSPACE).nodes`(노드 이름이 개체 이름)를 집합으로 만들어 `before_entities`에 담습니다.
2. `tracker.reset()` 뒤 `insert_path`의 본문을 `ids=insert_id`, `file_paths=insert_path.name`과 함께 `await rag.ainsert(...)`로 넣습니다. 한 편이라 목록 대신 값을 그대로 넘깁니다.
3. 추가에 든 호출 수 `tracker.call_count`를 `insert_calls`에 담습니다.
4. 추가 뒤 그래프의 개체 이름 집합에서 `before_entities`를 뺀 집합을 `new_entities`에 담습니다.

**반환값:** 이 셀이 만드는 변수 `before_entities`(set), `insert_calls`(int), `new_entities`(set)

In [ ]:
# [작성] 새 안내문을 문서 ID와 함께 추가하고 새로 생긴 개체를 찾으세요.
# 색인에서 빼 두었던 안내문(insert_id)의 파일 경로
insert_path = gwaje_dir / "docs" / f"{insert_id}.txt"

# 1. 추가 전 그래프의 개체 이름을 before_entities 집합에 담으세요.
...

# 2. 계수기를 0으로 되돌리고 insert_path의 본문을 문서 ID·파일 이름과 함께 추가하세요.
...

# 3. 갱신 호출 수를 insert_calls에 담으세요.
...

# 4. 추가 뒤 처음 생긴 개체 이름을 new_entities 집합에 담으세요.
...

In [ ]:
# [자가채점] 문서 상태와 그래프로 추가 결과를 확인합니다. 모델은 호출하지 않습니다.
inserted = await rag.aget_docs_by_ids(insert_id)
assert insert_id in inserted, "ainsert에 ids=insert_id를 넘겨 문서 ID로 찾을 수 있게 하세요."
assert inserted[insert_id]["status"] == "processed", "추가 문서 색인이 끝나지 않았습니다. 셀 출력의 오류를 확인하세요."
assert "insert_calls" in globals() and insert_calls != 0, "갱신 호출이 0회입니다. 같은 커널에서 추가 셀을 다시 실행하면 이미 넣은 문서로 보고 건너뜁니다. 커널을 다시 시작해 처음부터 실행하세요."
assert insert_calls < index_calls, "tracker.reset() 뒤 추가하고 tracker.call_count를 insert_calls에 담으세요. 새 문서만 추출하므로 색인보다 적습니다."
assert isinstance(before_entities, set) and isinstance(new_entities, set), "before_entities와 new_entities는 개체 이름 집합(set)입니다."
after_nodes = set(read_graph(WORKSPACE).nodes)
assert len(before_entities) < len(after_nodes), "before_entities는 추가하기 전에 모으세요."
assert new_entities and new_entities <= after_nodes and not (new_entities & before_entities), "new_entities에는 추가 뒤 그래프에서 before_entities를 뺀 개체만 담으세요."

print("문서 추가 확인 완료 | 갱신 호출:", insert_calls, "회 (색인", index_calls, "회)")
print("새로 생긴 개체:", sorted(new_entities))

새 개체와 기존 개체를 잇는 관계입니다. 새 제품이 기존 안내문의 어떤 증상·성분 개체에 붙었는지 보세요.

In [ ]:
# 한쪽 끝만 새 개체인 관계(새 문서가 기존 그래프에 붙은 곳)를 봅니다.
graph = read_graph(WORKSPACE)
link_rows = []

for source, target, edge in graph.edges(data=True):
    source_new = source in new_entities
    target_new = target in new_entities
    # 두 끝 가운데 하나만 새 개체이면 새 문서가 기존 개체에 이어진 관계입니다.
    if source_new != target_new:
        link_rows.append({"source": source, "target": target, "keywords": edge["keywords"]})

display(pd.DataFrame(link_rows))

## 7. 안내문을 지우고 M1을 다시 검색하세요

약국이 알레크롬점안액을 취급 목록에서 뺐습니다. 안내문이 남아 있으면 상담 도우미가 없는 제품을 권하게 됩니다. 교안에서 본 대로 `adelete_by_doc_id`는 문서와 그 청크, 그 문서에서만 나온 개체·관계를 지웁니다. 다른 문서에도 나온 개체는 남기고, 추출 응답 캐시에 남은 추출 결과로 설명을 다시 만듭니다. LLM은 설명이 많이 쌓인 개체를 다시 요약할 때만 부릅니다.

**작성:** 알레크롬점안액 안내문을 지우고, 그 제품이 답이던 질문 M1을 다시 검색하는 코드

**목적:** 교안의 문서 삭제를 적용해, 지운 문서가 검색 근거에서 빠지고 답이 바뀌는지 확인합니다.

**구현할 처리**
1. `tracker.reset()` 뒤 `delete_result = await rag.adelete_by_doc_id(delete_id)`로 지웁니다. `delete_id`와 `delete_question_id`(M1)는 셀 맨 위에서 읽어 둡니다.
2. 삭제에 든 호출 수 `tracker.call_count`를 `delete_calls`에 담습니다.
3. M1을 모드 비교 때와 같은 방법으로 다시 검색·답변해 결과를 `m1_after`에 담습니다. 질문 dict는 `questions_by_id[delete_question_id]`, `param`은 `QueryParam(mode="mix", chunk_top_k=TOP_K, enable_rerank=False, response_type=SHORT_ANSWER)`입니다. 부르기 전에 `tracker.reset()`하고 시간을 잽니다.
4. 모드 비교 때와 같은 모양의 `usage`를 만들고, `make_result_row("lightrag_mix_after_delete", 질문 dict, chunk_doc_ids(m1_after), usage, 걸린 초)`를 `m1_after_row`에 담습니다.

**반환값:** 이 셀이 만드는 변수 `delete_result`, `delete_calls`(int), `m1_after`(dict), `m1_after_row`(dict)

In [ ]:
# [작성] 안내문을 지우고 M1을 mix로 다시 검색하세요.
# 지울 문서 ID와 그 문서가 답이던 질문 ID
delete_id = extra["delete"]["doc_id"]
delete_question_id = extra["delete"]["question_id"]

# 1. 계수기를 0으로 되돌리고 delete_id 문서를 지워 결과를 delete_result에 담으세요.
...

# 2. 삭제 호출 수를 delete_calls에 담으세요.
...

# 3. M1을 mix 모드, 짧은 답으로 다시 검색·답변해 m1_after에 담으세요. 계수기와 시간도 잽니다.
...

# 4. method가 "lightrag_mix_after_delete"인 결과 한 행을 m1_after_row에 담으세요.
...

In [ ]:
# [자가채점] 삭제 결과와 다시 검색한 근거를 확인합니다. 모델은 호출하지 않습니다.
assert "delete_result" in globals() and delete_result.status == "success", "adelete_by_doc_id(delete_id)의 결과를 delete_result에 담으세요. 상태가 success가 아니면 셀 출력을 확인하세요."
assert await rag.aget_docs_by_ids(delete_id) == {}, "delete_id 문서가 아직 남아 있습니다."
assert "delete_calls" in globals() and isinstance(delete_calls, int), "tracker.call_count를 delete_calls에 담으세요."
assert isinstance(m1_after, dict) and "llm_response" in m1_after, "aquery_llm의 결과를 m1_after에 담으세요."
assert delete_id not in chunk_doc_ids(m1_after), "지운 문서의 청크가 검색 결과에 남아 있습니다."
assert m1_after_row["method"] == "lightrag_mix_after_delete" and m1_after_row["question_id"] == delete_question_id, "M1 질문으로 method가 'lightrag_mix_after_delete'인 행을 만드세요."
assert 1 <= m1_after_row["llm_calls"] <= 3, "다시 검색하기 전에 tracker.reset()을 하세요."

print("문서 삭제 확인 완료 | 삭제 호출:", delete_calls, "회")

삭제 전후 M1의 결과 행과 답변을 비교합니다. 이연제약 개체가 남았는지도 봅니다.

In [ ]:
# 삭제 전 mix 결과 행과 삭제 뒤 M1 결과 행
m1_before_row = results[(results["question_id"] == delete_question_id) & (results["method"] == "lightrag_mix")]

# 두 행을 위아래로 붙여 hit_docs와 docs_given이 어떻게 바뀌었는지 비교합니다.
display(pd.concat([m1_before_row.drop(columns="hit_rate"), pd.DataFrame([m1_after_row])], ignore_index=True))

In [ ]:
# 삭제 뒤 M1 답변과 남은 이연제약 개체
print("이연제약 개체:", [name for name in read_graph(WORKSPACE).nodes if "이연제약" in name])
display(Markdown(m1_after["llm_response"]["content"]))

In [ ]:
# 메모리에 있는 그래프와 벡터를 파일에 저장하고 저장소를 닫습니다. 닫지 않으면 마지막 변경이 파일에 남지 않을 수 있습니다.
await rag.finalize_storages()

## 결과를 읽고 정리하세요

**작성:** 서술형 답안. 숫자는 자신이 실행한 결과 표에서 인용하세요.

**목적:** 교안에서 배운 모드별 검색 방식과 문서 추가·삭제를, 이 자료의 결과 표와 답변으로 판단합니다.

**설명할 내용**
1. 질문 유형별 요약에서 `naive`와 `mix`의 근거 문서 포함률과 질문 1개당 LLM 호출·입력 토큰을 비교하세요. 멀티홉 질문에서 차이가 난(또는 나지 않은) 이유를 개체 유형 안내에 넣은 제조사·성분과 연결해 설명하세요.
2. G1·G2는 두 모드 모두 `all_hit`가 0입니다. 그 이유를 `TOP_K`와 근거 문서 수로 설명하고, 빠짐없이 답하려면 무엇을 바꿔야 할지 쓰세요.
3. 더모픽스크림 추가 호출 수를 색인 호출 수와 비교하고, 새 개체가 기존 그래프의 어떤 개체에 붙었는지 쓰세요.
4. 알레크롬점안액을 지운 뒤 M1의 근거와 답이 어떻게 바뀌었는지, 이연제약 개체는 왜 남았는지 설명하세요.

*(여기에 확인한 근거와 판단 이유를 작성하세요.)*